# 🧠 nrjNET — My First Neural Network, From Scratch

This notebook builds a tiny neural network using **nothing but plain Python** — no PyTorch magic.
Every number, every gradient, every weight update is something we write ourselves.

Inspired by Andrej Karpathy's video *"The spelled-out intro to neural networks and backpropagation: building micrograd"*.

### 🗺️ Roadmap

| Part | What we learn | Key idea |
|---|---|---|
| **1** | Derivatives | *"If I nudge the input a tiny bit, how much does the output move?"* |
| **2** | The `Value` class | A number that remembers how it was made |
| **3** | Backpropagation | The chain rule, done automatically |
| **4** | Checking with PyTorch | Proof that our gradients are correct |
| **5** | Neuron → Layer → MLP | Building the network |
| **6** | Training | Loss, gradient descent, and watching it learn |
| **7** | Recap | Lessons and mistakes to remember |

> 💡 **How to use this notebook:** always run cells **top to bottom** (`Runtime → Run all` in Colab).
> Running cells out of order can leave old variables in memory and give confusing results.

In [ ]:
import math                      # exp, tanh, etc. for single numbers
import random                    # random starting weights for the network
import numpy as np               # only used in Part 1, to evaluate f on many x values at once
import matplotlib.pyplot as plt  # plotting

# Fix the random seed so the "random" weights are the same every run.
# This makes results reproducible: same notebook -> same numbers.
random.seed(42)

---
# Part 1 — What is a derivative, really?

Forget the calculus formulas for a moment. A derivative answers one simple question:

> **If I nudge the input by a tiny amount `h`, how much does the output change — per unit of nudge?**

$$\text{slope} \approx \frac{f(x + h) - f(x)}{h}$$

This is the **only** idea a neural network needs. Training = repeatedly asking
*"which direction should I nudge each weight to make the error smaller?"*

Let's start with a simple curve: $f(x) = 3x^2 - 4x + 5$

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

f(3.0)   # 3*9 - 12 + 5 = 20.0

In [ ]:
# Plot f(x) for x from -5 to 5 so we can SEE the curve.
xs = np.arange(-5, 5, 0.25)   # [-5.0, -4.75, ..., 4.75]
ys = f(xs)                    # numpy applies f to every x at once

plt.plot(xs, ys)
plt.title("f(x) = 3x² - 4x + 5")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.grid(True)
plt.show()

### Measuring the slope numerically

Instead of differentiating with pen and paper, we just **nudge and measure**.

In [ ]:
def numerical_derivative(f, x, h=0.001):
    # Nudge x by a tiny h, see how much f moved, divide by h.
    return (f(x + h) - f(x)) / h

print(numerical_derivative(f, -3.0))   # ≈ -22  -> curve going DOWN steeply on the left
print(numerical_derivative(f,  3.0))   # ≈ +14  -> curve going UP on the right
print(numerical_derivative(f, 2/3))    # ≈ 0    -> the bottom of the bowl (flat)

# Check with calculus: f'(x) = 6x - 4  ->  f'(-3) = -22,  f'(3) = 14,  f'(2/3) = 0  ✅

> 🔑 **Sign tells direction, size tells steepness.**
> - Negative slope → increasing `x` makes `f` **smaller**
> - Positive slope → increasing `x` makes `f` **bigger**
> - Zero slope → we're at a flat spot (often a minimum — exactly what training looks for!)

### More than one input

A neural network has **many** inputs (weights). We ask the same question for each one separately:
*"If I nudge only `a`, how does `d` change?"* This is called a **partial derivative**.

In [ ]:
a, b, c = 2.0, -3.0, 10.0
h = 0.0001

d1 = a*b + c          # original output: 2*(-3) + 10 = 4.0

# Nudge ONLY a, keep b and c fixed
d2 = (a + h)*b + c
print("slope of d w.r.t. a:", (d2 - d1) / h)   # = b = -3   (making a bigger makes d smaller)

# Nudge ONLY b
d2 = a*(b + h) + c
print("slope of d w.r.t. b:", (d2 - d1) / h)   # = a = 2

# Nudge ONLY c
d2 = a*b + (c + h)
print("slope of d w.r.t. c:", (d2 - d1) / h)   # = 1    (c is added directly)

> 📝 Notice the pattern — we'll use it in Part 2:
> - For **multiplication** `a*b`: the slope w.r.t. `a` is just **the other number** (`b`).
> - For **addition** `a + c`: the slope is always **1** — addition just passes changes straight through.

---
# Part 2 — The `Value` class: a number with a memory

Nudging every input one by one is slow (a real network has millions of weights).
Instead we build a smarter number, `Value`, that **remembers how it was created**.

Every time we do `a * b` or `a + b`, the result keeps a note:
*"I was made by `*` from `a` and `b`."* This builds a **graph** of the whole calculation.

Then, by walking that graph **backwards**, we can get every derivative in one sweep.
That backward walk is **backpropagation**.

Each `Value` stores:

| Attribute | Meaning |
|---|---|
| `data` | the actual number |
| `grad` | ∂(final output)/∂(this value) — *"how much does the final answer change if I nudge this?"* |
| `_prev` | the Values it was made from (its "children" in the graph) |
| `_op` | the operation that made it (`+`, `*`, `tanh`…) |
| `_backward` | a small function that passes the gradient to its children (the chain rule) |

In [ ]:
class Value:
    """A single number that remembers how it was computed, so it can do backprop."""

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data                  # the actual number, e.g. 2.0
        self.grad = 0.0                   # d(output)/d(this). 0 = "no effect yet" until backward() runs
        self._backward = lambda: None     # how to push my grad to my children (set by each operation)
        self._prev = set(_children)       # the Values that were combined to make me
        self._op = _op                    # the operation that made me ('+', '*', ...) - for drawing
        self.label = label                # an optional name - for drawing

    def __repr__(self):
        # What gets printed when you print a Value
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ---------------- Basic operations ----------------
    # Each operation does TWO things:
    #   1. FORWARD : compute the result number
    #   2. BACKWARD: define how the gradient flows back to the inputs (local derivative × out.grad)

    def __add__(self, other):                                   # self + other
        other = other if isinstance(other, Value) else Value(other)   # allow Value + 2.0
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # d(a+b)/da = 1 and d(a+b)/db = 1  -> addition passes the gradient through unchanged
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):                                   # self * other
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(a*b)/da = b and d(a*b)/db = a  -> each input gets "the other one" × out.grad
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):                                   # self ** number
        assert isinstance(other, (int, float)), "only int/float powers are supported"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            # Power rule: d(x^n)/dx = n * x^(n-1)
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    # ---------------- Activation functions ----------------

    def tanh(self):
        # tanh squashes any number into the range (-1, 1).
        # Formula: (e^(2x) - 1) / (e^(2x) + 1). We use math.tanh because the formula
        # overflows for big x (e^(2x) gets too large for a float), math.tanh does not.
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d(tanh(x))/dx = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')

        def _backward():
            # d(e^x)/dx = e^x  (e^x is its own derivative!)
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    # ---------------- Convenience operations ----------------
    # These are built from the ones above, so they get backprop for free.

    def __neg__(self):          return self * -1             # -self
    def __sub__(self, other):   return self + (-other)       # self - other
    def __truediv__(self, other): return self * other**-1    # self / other  (a/b = a * b^-1)

    # The "r" versions handle a number on the LEFT, e.g. 2 * Value. Python tries
    # int.__mul__ first, fails, then calls Value.__rmul__.
    def __radd__(self, other):  return self + other          # other + self
    def __rmul__(self, other):  return self * other          # other * self
    def __rsub__(self, other):  return other + (-self)       # other - self
    def __rtruediv__(self, other): return other * self**-1   # other / self

    # ---------------- Backpropagation ----------------

    def backward(self):
        """Compute the gradient of THIS value with respect to every Value that led to it."""

        # Step 1: put all nodes in "topological order" - every node comes AFTER the
        # nodes it was made from. Walking this list in reverse visits each node only
        # after its gradient is complete.
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Step 2: d(self)/d(self) = 1. This is where the gradient starts.
        self.grad = 1.0

        # Step 3: go from the output back to the inputs, applying the chain rule at each node.
        for node in reversed(topo):
            node._backward()

> ⚠️ **Why `+=` and not `=` in every `_backward`?**
> If a Value is used **twice** (e.g. `b = a + a`), gradient flows back to it along **both** paths,
> and the contributions must be **added**. With `=`, the second path would overwrite the first.
> This is also why we must **reset grads to 0** before each new backward pass (see Part 6).

### Drawing the graph

This helper draws the computation graph so we can *see* data and gradients flowing.
Each box shows `label | data | grad`. (In Colab this works out of the box. Locally you need
`pip install graphviz` **and** the Graphviz app from graphviz.org.)

In [ ]:
try:
    from graphviz import Digraph
except ImportError:
    Digraph = None
    print("graphviz not installed - graphs will be skipped. (pip install graphviz)")


def trace(root):
    """Collect every node and edge in the graph that ends at `root`."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges


def draw_dot(root):
    """Draw the computation graph, left (inputs) to right (output)."""
    if Digraph is None:
        return None
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # A rectangle for every Value showing its label, data and grad
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # A small circle for the operation that produced this Value
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # Connect each input Value to the operation that used it
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

### Trying it out

Same expression as Part 1, `d = a*b + c`, but now with `Value`s.

In [ ]:
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b;  e.label = 'e'      # e = -6
d = e + c;  d.label = 'd'      # d = 4

d.backward()                   # fill in every .grad automatically

print("d     =", d.data)
print("a.grad =", a.grad, " (Part 1 measured -3)")
print("b.grad =", b.grad, "  (Part 1 measured  2)")
print("c.grad =", c.grad, "  (Part 1 measured  1)")

In [ ]:
draw_dot(d)
# Read it right-to-left: d.grad = 1 (start), '+' copies it to e and c,
# '*' gives a the other input (b = -3) and b the other input (a = 2).

🎉 **Same answers as nudging by hand in Part 1 — but computed in a single backward pass**,
no matter how many inputs there are.

---
# Part 3 — Backpropagation & the chain rule, on a real neuron

A neuron takes inputs `x`, multiplies each by a weight `w`, adds a bias `b`,
and squashes the result with an **activation function** (here `tanh`):

$$o = \tanh(x_1 w_1 + x_2 w_2 + b)$$

```
x1 ──(×w1)──┐
            ├──(+)──(+b)── n ──tanh── o
x2 ──(×w2)──┘
```

In [ ]:
# Inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0,  label='w2')
# Bias (this odd number just makes the output a nice 0.7071)
b = Value(6.8813735870195432, label='b')

# Forward pass, one step at a time so every node gets a name in the graph
x1w1 = x1 * w1;          x1w1.label = 'x1*w1'
x2w2 = x2 * w2;          x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2;  x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b;        n.label = 'n'
o = n.tanh();            o.label = 'o'

# Backward pass
o.backward()

print(f"output o = {o.data:.4f}")
for v in [x1, w1, x2, w2, b]:
    print(f"{v.label:>2}.grad = {v.grad: .4f}")

In [ ]:
draw_dot(o)

### How does backprop get `x1.grad`? The chain rule.

Take the multiplication node $z = x_1 \cdot w_1$. We want $\dfrac{\partial o}{\partial x_1}$ —
how the **final output** changes when we nudge $x_1$.

$x_1$ doesn't affect $o$ directly; it affects $z$, which then affects $o$. The **chain rule** says
just **multiply the two effects**:

$$\frac{\partial o}{\partial x_1} \;=\; \underbrace{\frac{\partial o}{\partial z}}_{\text{gradient arriving from the right}} \;\times\; \underbrace{\frac{\partial z}{\partial x_1}}_{\text{local derivative}}$$

- **Gradient arriving from the right**, $\frac{\partial o}{\partial z}$: already computed and stored in `x1w1.grad`.
- **Local derivative**, $\frac{\partial z}{\partial x_1}$: since $z = x_1 w_1$, it's just $w_1$, stored in `w1.data`.

So:

$$\texttt{x1.grad} = \texttt{w1.data} \times \texttt{x1w1.grad} = (-3) \times 0.5 = -1.5 \;✅$$

That is **exactly** the line `self.grad += other.data * out.grad` inside `__mul__`.
Backprop is just this rule, applied node by node from the output back to the inputs.

> 🧩 Interesting: `w2.grad = 0` because `x2 = 0`. If the input is zero, changing its weight has no effect.

### Sanity check: does backprop agree with nudging?

Let's verify `w1.grad` the Part 1 way — nudge `w1`, rerun the forward pass, measure the slope.

In [ ]:
def neuron(x1, x2, w1, w2, b):
    # Same neuron, plain floats (no Value), just for checking
    return math.tanh(x1*w1 + x2*w2 + b)

h = 1e-6
base   = neuron(2.0, 0.0, -3.0,     1.0, 6.8813735870195432)
nudged = neuron(2.0, 0.0, -3.0 + h, 1.0, 6.8813735870195432)

print("w1.grad by nudging :", round((nudged - base) / h, 4))
print("w1.grad by backprop:", round(w1.grad, 4))

---
# Part 4 — Checking our work with PyTorch

PyTorch is a professional deep-learning library. It does **exactly what our `Value` class does**,
just on big arrays ("tensors") and much faster. If our gradients match PyTorch's, our engine is correct.

| Our code | PyTorch |
|---|---|
| `Value(2.0)` | `torch.tensor([2.0], requires_grad=True)` |
| `o.backward()` | `o.backward()` |
| `x1.grad` | `x1.grad.item()` |

*(PyTorch is pre-installed in Colab. Locally: `pip install torch`.)*

In [ ]:
try:
    import torch

    # .double() = 64-bit floats, same precision as Python floats, so results match exactly.
    # requires_grad=True tells PyTorch "track this, I want its gradient".
    x1_t = torch.tensor([2.0]).double().requires_grad_(True)
    x2_t = torch.tensor([0.0]).double().requires_grad_(True)
    w1_t = torch.tensor([-3.0]).double().requires_grad_(True)
    w2_t = torch.tensor([1.0]).double().requires_grad_(True)
    b_t  = torch.tensor([6.8813735870195432]).double().requires_grad_(True)

    o_t = torch.tanh(x1_t*w1_t + x2_t*w2_t + b_t)   # forward pass
    o_t.backward()                                  # backward pass

    print(f"{'':4}{'PyTorch':>10}{'nrjNET':>10}")
    print(f"{'o':4}{o_t.item():>10.4f}{o.data:>10.4f}")
    for name, t, v in [('x1', x1_t, x1), ('w1', w1_t, w1), ('x2', x2_t, x2), ('w2', w2_t, w2)]:
        print(f"{name:4}{t.grad.item():>10.4f}{v.grad:>10.4f}")
except ImportError:
    print("PyTorch not installed - skipping this comparison. (pip install torch)")

---
# Part 5 — Building the network: Neuron → Layer → MLP

Now we stack neurons like LEGO:

- **Neuron**: weights + bias → `tanh(w·x + b)` → one number
- **Layer**: a group of neurons that all see the **same** inputs → a list of numbers
- **MLP** (Multi-Layer Perceptron): layers in a chain — each layer's output is the next layer's input

```
inputs (3)     layer 1 (4)     layer 2 (4)     output (1)
   x1  ───────►   ○   ───────►   ○
   x2  ───────►   ○   ───────►   ○   ───────►   ○  → prediction
   x3  ───────►   ○   ───────►   ○
                  ○              ○
```

In [ ]:
class Neuron:
    """One neuron: out = tanh(w1*x1 + w2*x2 + ... + b)"""

    def __init__(self, nin):
        # nin = number of inputs. One weight per input, plus one bias.
        # Start them random in [-1, 1]. If all weights started equal, every neuron would
        # learn the same thing - randomness lets them specialise.
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # __call__ lets us write neuron(x) as if the neuron were a function.
        # zip pairs each weight with its input: (w1,x1), (w2,x2), ...
        # sum(..., self.b) starts the sum at b instead of 0, so we get w·x + b.
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        # Everything training is allowed to change
        return self.w + [self.b]


class Layer:
    """A row of `nout` neurons, each looking at the same `nin` inputs."""

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        # If the layer has one neuron, return the Value itself instead of a 1-item list
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    """Multi-Layer Perceptron: layers connected one after another."""

    def __init__(self, nin, nouts):
        # Example: MLP(3, [4, 4, 1]) -> sizes [3, 4, 4, 1] -> layers 3→4, 4→4, 4→1
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        # Feed forward: output of each layer becomes input of the next
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
# nrjNET: 3 inputs -> 4 neurons -> 4 neurons -> 1 output
nrjnet = MLP(3, [4, 4, 1])

x = [2.0, 3.0, -1.0]
print("prediction for x:", nrjnet(x))
print("number of parameters:", len(nrjnet.parameters()))
# Layer 1: 4 neurons × (3 weights + 1 bias) = 16
# Layer 2: 4 neurons × (4 weights + 1 bias) = 20
# Layer 3: 1 neuron  × (4 weights + 1 bias) =  5
#                                      total = 41

In [ ]:
draw_dot(nrjnet(x))
# A big graph! Every one of these nodes gets a gradient in a single backward pass.

---
# Part 6 — Training nrjNET

### The dataset
Four examples, each with 3 input numbers and a target answer of `+1` or `-1`.
The network's job is to learn to output the right target for each input.

In [ ]:
xs = [
    [2.0,  3.0, -1.0],   # target  1.0
    [3.0, -1.0,  0.5],   # target -1.0
    [0.5,  1.0,  1.0],   # target -1.0
    [1.0,  1.0, -1.0],   # target  1.0
]
ys = [1.0, -1.0, -1.0, 1.0]   # the correct answers ("ground truth")

# Before training the predictions are basically random:
ypred = [nrjnet(x) for x in xs]
for target, pred in zip(ys, ypred):
    print(f"target {target:+.1f}   prediction {pred.data:+.4f}")

### The loss: one number that says "how wrong are we?"

For each example, take `(prediction - target)²`, then add them all up:

$$\text{Loss} = \sum_i (\hat{y}_i - y_i)^2$$

- Squaring makes every error positive (being wrong by -0.5 is as bad as +0.5) and punishes big errors more.
- **Loss = 0 means perfect predictions.** Training = making this number as small as possible.

*(This is the "sum of squared errors". Divide by the number of examples and you get the
"mean squared error" (MSE). Both work the same for training.)*

### Gradient descent: the training recipe

Repeat:
1. **Forward pass**: compute predictions and the loss
2. **Zero the gradients**: clear the old `.grad` values (remember the `+=` in Part 2!)
3. **Backward pass**: `loss.backward()` fills every parameter's `.grad`
4. **Update**: nudge each parameter **against** its gradient:  `p.data -= learning_rate * p.grad`

> 🔑 **Why minus?** `p.grad` points in the direction that **increases** the loss.
> We want the loss to go **down**, so we step the **opposite** way.
> Using `+=` here would climb uphill and make the network *worse*.

In [ ]:
learning_rate = 0.05   # how big each step is. Too big -> overshoots, too small -> very slow.
losses = []            # keep a history so we can plot it

for step in range(100):

    # 1. Forward pass
    ypred = [nrjnet(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # 2. Zero the gradients (otherwise they pile up from the previous step)
    for p in nrjnet.parameters():
        p.grad = 0.0

    # 3. Backward pass
    loss.backward()

    # 4. Update: step every parameter downhill
    for p in nrjnet.parameters():
        p.data -= learning_rate * p.grad

    losses.append(loss.data)
    if step % 10 == 0:
        print(f"step {step:3d}   loss {loss.data:.6f}")

print(f"final loss {losses[-1]:.6f}")

In [ ]:
plt.plot(losses)
plt.title("nrjNET learning: loss over training steps")
plt.xlabel("training step"); plt.ylabel("loss")
plt.grid(True)
plt.show()

In [ ]:
# After training: predictions should be very close to the targets
ypred = [nrjnet(x) for x in xs]
for target, pred in zip(ys, ypred):
    print(f"target {target:+.1f}   prediction {pred.data:+.4f}")

---
# Part 7 — Recap 🎓

**What I built:**
1. A **derivative** is "nudge the input, measure the output".
2. The **`Value`** class remembers how each number was made, building a graph.
3. **Backprop** walks that graph backwards, applying the **chain rule** (`local derivative × incoming gradient`) at every node.
4. **PyTorch** gives the same gradients, so the engine is correct.
5. **Neuron → Layer → MLP** stacks the simple pieces into a network.
6. **Training** = forward → zero grads → backward → step downhill, repeated.

### 🐛 Mistakes from my first messy version (so I don't repeat them)

| Mistake | What happened | Fix |
|---|---|---|
| `p.data += lr * p.grad` | Climbed **up** the loss: weights blew up to -34, tanh got stuck at 1.0, loss ≈ 8 (the worst possible) | Use `-=`: always step **against** the gradient |
| Forgot to zero grads | Gradients from old steps pile up because of `+=` | `p.grad = 0.0` for **every** parameter before `backward()` |
| Running cells out of order | Printed `ypred` before it was computed, so it showed old values | Always **Run all** top to bottom |

### 🚀 Next ideas to try
- Change the network size: `MLP(3, [8, 8, 1])`. Does it learn faster?
- Try `learning_rate = 0.5` or `0.001`. What happens to the loss curve?
- Add a `relu()` method to `Value` (`max(0, x)`, derivative 1 if x > 0 else 0) and use it instead of `tanh`.